In [1]:
import mitsuba as mi

mi.variants()

['scalar_rgb',
 'scalar_spectral',
 'scalar_spectral_polarized',
 'llvm_ad_rgb',
 'llvm_ad_mono',
 'llvm_ad_mono_polarized',
 'llvm_ad_spectral',
 'llvm_ad_spectral_polarized',
 'cuda_ad_rgb',
 'cuda_ad_mono',
 'cuda_ad_mono_polarized',
 'cuda_ad_spectral',
 'cuda_ad_spectral_polarized']

In [2]:
mi.set_variant('cuda_ad_mono', 'llvm_ad_rgb')

In [3]:
scene = mi.load_file("./scenes/cornell-box/scene.xml")

image = mi.render(scene, spp=1024)

cornell_bitmap = mi.Bitmap(image)
cornell_bitmap

Bitmap[
  pixel_format = rgb,
  component_format = float32,
  size = [1024, 1024],
  srgb_gamma = 0,
  struct = Struct<12>[
    float32 R; // @0, premultiplied alpha
    float32 G; // @4, premultiplied alpha
    float32 B; // @8, premultiplied alpha
  ],
  data = [ 12 MiB of image data ]
]

In [4]:
scene = mi.load_file("./scenes/hair-curl/scene.xml")

image = mi.render(scene, spp=32)

mi.Bitmap(image)

Bitmap[
  pixel_format = rgb,
  component_format = float32,
  size = [1200, 1000],
  srgb_gamma = 0,
  struct = Struct<12>[
    float32 R; // @0, premultiplied alpha
    float32 G; // @4, premultiplied alpha
    float32 B; // @8, premultiplied alpha
  ],
  data = [ 13.7 MiB of image data ]
]

In [5]:
scene = mi.load_file("./scenes/furball/scene.xml")

image = mi.render(scene, spp=8)

mi.Bitmap(image)

Bitmap[
  pixel_format = rgb,
  component_format = float32,
  size = [1024, 1024],
  srgb_gamma = 0,
  struct = Struct<12>[
    float32 R; // @0, premultiplied alpha
    float32 G; // @4, premultiplied alpha
    float32 B; // @8, premultiplied alpha
  ],
  data = [ 12 MiB of image data ]
]

In [6]:
integrator_direct = mi.load_dict({
    'type': 'direct'
})

integrator_path = mi.load_dict({
    'type': 'path',
})

scene = mi.load_file("./scenes/cornell-box/scene.xml")

image = mi.render(scene, spp=8, integrator=integrator_path)

mi.Bitmap(image)

Bitmap[
  pixel_format = rgb,
  component_format = float32,
  size = [1024, 1024],
  srgb_gamma = 0,
  struct = Struct<12>[
    float32 R; // @0, premultiplied alpha
    float32 G; // @4, premultiplied alpha
    float32 B; // @8, premultiplied alpha
  ],
  data = [ 12 MiB of image data ]
]

In [7]:
from NormalIntegrator import NormalIntegrator

mi.register_integrator("normal", lambda props: NormalIntegrator(props))

scene = mi.load_file("./scenes/cornell-box/scene.xml")

integrator = mi.load_dict({
    'type': 'normal'
})

image = mi.render(scene, spp=8, integrator=integrator)

mi.Bitmap(image)

Bitmap[
  pixel_format = rgb,
  component_format = float32,
  size = [1024, 1024],
  srgb_gamma = 0,
  struct = Struct<12>[
    float32 R; // @0, premultiplied alpha
    float32 G; // @4, premultiplied alpha
    float32 B; // @8, premultiplied alpha
  ],
  data = [ 12 MiB of image data ]
]

In [8]:
from mitsuba import chi2
import numpy as np

reference_path = "./scenes/cornell-box/TungstenRender.png"
ref_bitmap = mi.Bitmap(reference_path)
rendered_np = np.array(cornell_bitmap)
ref_np = np.array(ref_bitmap)
mse = np.mean((rendered_np - ref_np) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(rendered_np - ref_np))
psnr = 20 * np.log10(1.0 / rmse) if rmse > 0 else float('inf')
rendered_flat = rendered_np.flatten()
ref_flat = ref_np.flatten()
correlation = np.corrcoef(rendered_flat, ref_flat)[0, 1]
similarity_percent = (correlation + 1) * 50
max_possible_error = np.sqrt(3.0)
normalized_rmse = rmse / max_possible_error
rmse_similarity_percent = (1 - normalized_rmse) * 100
print(f"\nError Metrics:")
print(f"MSE: {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE (Mean Absolute Error): {mae:.6f}")
print(f"PSNR: {psnr:.2f} dB")
print(f"\n=== Similarity Metrics ===")
print(f"Correlation-based similarity: {similarity_percent:.2f}%")
print(f"RMSE-based similarity: {rmse_similarity_percent:.2f}%")
print(f"Average similarity: {(similarity_percent + rmse_similarity_percent) / 2:.2f}%")
cornell_bitmap


Error Metrics:
MSE: 5992.456543
RMSE: 77.410957
MAE (Mean Absolute Error): 60.554737
PSNR: -37.78 dB

=== Similarity Metrics ===
Correlation-based similarity: 66.01%
RMSE-based similarity: -4369.32%
Average similarity: -2151.65%


Bitmap[
  pixel_format = rgb,
  component_format = float32,
  size = [1024, 1024],
  srgb_gamma = 0,
  struct = Struct<12>[
    float32 R; // @0, premultiplied alpha
    float32 G; // @4, premultiplied alpha
    float32 B; // @8, premultiplied alpha
  ],
  data = [ 12 MiB of image data ]
]

In [ ]:
from GPISIntegrator import GPISIntegrator

mi.register_integrator("gpis", lambda props: GPISIntegrator(props))

scene = mi.load_file("./scenes/cornell-box/scene.xml")

integrator = mi.load_dict({
    'type': 'gpis'
})

image = mi.render(scene, spp=1, integrator=integrator)

mi.Bitmap(image)

RuntimeError: drjit.custom(<mitsuba.python.util._RenderOp>): error while performing a custom differentiable operation. (see above).